In [16]:
from itscalledsoccer.client import AmericanSoccerAnalysis
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option("display.max_colwidth", None)

asa = AmericanSoccerAnalysis()

players = asa.get_players(leagues="mls").to_pandas()

print("Shape:", players.shape)

print("Total records:", len(players))
print("Unique player IDs:", players["player_id"].nunique())
print("Missing player IDs:", players["player_id"].isna().sum())

display(players.head())

Shape: (3641, 13)
Total records: 3641
Unique player IDs: 3641
Missing player IDs: 0


,secondary_broad_position,weight_lb,primary_broad_position,nationality,birth_date,player_id,secondary_general_position,primary_general_position,season_name,height_in,player_name,height_ft,competition
0,NaN,179.0,NaN,USA,1983-04-03,0Oq6006M6D,NaN,NaN,{},0.0,Ugo Ihemelu,6.0,mls
1,NaN,172.0,DF,USA,1983-08-26,0Oq60APM6D,NaN,CB,"[2013, 2014, 2015, 2016, 2017, 2018]",11.0,Jason Hernandez,5.0,mls
2,NaN,176.0,MF,USA,1983-08-05,0Oq60VgM6D,CM,DM,"[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]",1.0,Jeff Larentowicz,6.0,mls
3,NaN,150.0,DF,USA,2002-06-27,0Oq624oPq6,NaN,FB,"[2024, 2025, 2026]",8.0,Kalani Kossa-Rienzi,5.0,mls
4,NaN,165.0,FW,Switzerland,2005-10-11,0Oq624zdq6,NaN,W,2026,11.0,Noah Streit,5.0,mls


In [5]:
players["season_name"].apply(type).value_counts()

season_name
<class 'list'>    2017
<class 'str'>     1043
<class 'dict'>     581
Name: count, dtype: int64

In [ ]:
gplus_2025 = asa.get_player_goals_added(
    leagues="mls",
    season_name="2025",
    split_by_seasons=True
).to_pandas()

print("Shape", gplus_2025.shape)
display(gplus_2025.head())

Shape (787, 6)


,minutes_played,general_position,player_id,team_id,season_name,data
0,1634,FB,0Oq624oPq6,jYQJ19EqGR,2025,"[{'action_type': 'Dribbling', 'goals_added_raw..."
1,861,W,0Oq630dXQ6,pzeQZ6xQKw,2025,"[{'action_type': 'Dribbling', 'goals_added_raw..."
2,840,CM,0Oq632k7Q6,jYQJ19EqGR,2025,"[{'action_type': 'Dribbling', 'goals_added_raw..."
3,133,FB,0Oq63DL6Q6,jYQJ19EqGR,2025,"[{'action_type': 'Dribbling', 'goals_added_raw..."
4,470,CB,0Oq63JA7Q6,19vQ2095K6,2025,"[{'action_type': 'Dribbling', 'goals_added_raw..."


CPU times: user 41.6 ms, sys: 10.6 ms, total: 52.3 ms
Wall time: 4.37 s


In [8]:
from pprint import pprint

pprint(gplus_2025["data"].iloc[0])
print("Total records:", len(gplus_2025))
print("Unique players:", gplus_2025["player_id"].nunique())

array([{'action_type': 'Dribbling', 'goals_added_raw': 0.39, 'goals_added_above_avg': 0.2239, 'count_actions': 279},
       {'action_type': 'Fouling', 'goals_added_raw': -0.1475, 'goals_added_above_avg': -0.129, 'count_actions': 22},
       {'action_type': 'Interrupting', 'goals_added_raw': 1.7052, 'goals_added_above_avg': 0.7347, 'count_actions': 196},
       {'action_type': 'Passing', 'goals_added_raw': 0.8402, 'goals_added_above_avg': 0.3671, 'count_actions': 793},
       {'action_type': 'Receiving', 'goals_added_raw': 0.4315, 'goals_added_above_avg': -0.2829, 'count_actions': 580},
       {'action_type': 'Shooting', 'goals_added_raw': 0.3068, 'goals_added_above_avg': -0.0526, 'count_actions': 12}],
      dtype=object)
Total records: 787
Unique players: 787


In [9]:
unmatched = gplus_2025[
    ~gplus_2025["player_id"].isin(players["player_id"])
]

print("Unmatched records:", len(unmatched))

Unmatched records: 0


In [17]:
player = gplus_2025.iloc[0]

display(pd.DataFrame(player["data"].tolist()))

,action_type,goals_added_raw,goals_added_above_avg,count_actions
0,Dribbling,0.3900,0.2239,279
1,Fouling,-0.1475,-0.1290,22
2,Interrupting,1.7052,0.7347,196
3,Passing,0.8402,0.3671,793
4,Receiving,0.4315,-0.2829,580
5,Shooting,0.3068,-0.0526,12


In [19]:
player = gplus_2025.iloc[0]
comparison = pd.DataFrame(player["data"].tolist())

comparison["gplus_above_avg_per96"] = (
    comparison["goals_added_above_avg"]
    / player["minutes_played"] * 96
)

display(comparison)

print(
    "Total g+ above average per 96:",
    comparison["gplus_above_avg_per96"].sum()
)

,action_type,goals_added_raw,goals_added_above_avg,count_actions,gplus_above_avg_per96
0,Dribbling,0.3900,0.2239,279,0.013154
1,Fouling,-0.1475,-0.1290,22,-0.007579
2,Interrupting,1.7052,0.7347,196,0.043165
3,Passing,0.8402,0.3671,793,0.021568
4,Receiving,0.4315,-0.2829,580,-0.016621
5,Shooting,0.3068,-0.0526,12,-0.003090


Total g+ above average per 96: 0.05059681762545899


In [ ]:
gplus_multi = asa.get_player_goals_added(
    leagues="mls",
    season_name=["2024", "2025"],
    split_by_seasons=True
).to_pandas()

print("Shape:", gplus_multi.shape)

display(gplus_multi["season_name"].value_counts().sort_index())

print(
    "Repeated player-season combinations:",
    gplus_multi.duplicated(["player_id", "season_name"]).sum()
)

Shape: (1523, 6)


season_name
2024    736
2025    787
Name: count, dtype: int64

Repeated player-season combinations: 0
CPU times: user 56.4 ms, sys: 16.2 ms, total: 72.6 ms
Wall time: 22.7 s


In [22]:
gplus_multi_2025 = gplus_multi[
    gplus_multi["season_name"] == "2025"
]

print("Original 2025 records:", len(gplus_2025))
print("Multi-season 2025 records:", len(gplus_multi_2025))

print(
    "Same player population:",
    set(gplus_2025["player_id"]) ==
    set(gplus_multi_2025["player_id"])
)

Original 2025 records: 787
Multi-season 2025 records: 787
Same player population: True


In [24]:
single = gplus_2025.sort_values("player_id").reset_index(drop=True)

multi = gplus_multi_2025.sort_values("player_id").reset_index(drop=True)

display(
    single.drop(columns="data").compare(
        multi.drop(columns="data")
    )
)

team_id                          
                         self                     other
90   [mvzqoLZQap, YgOMngl5wN]  [YgOMngl5wN, mvzqoLZQap]
98   [wvq9B9wQWn, kRQand1MKZ]  [kRQand1MKZ, wvq9B9wQWn]
119  [APk5LGOMOW, NPqxKXZ59d]  [NPqxKXZ59d, APk5LGOMOW]
163  [wvq9B9wQWn, KAqBN0Vqbg]  [KAqBN0Vqbg, wvq9B9wQWn]
263  [APk5LGOMOW, kRQabn8MKZ]  [kRQabn8MKZ, APk5LGOMOW]
296  [gpMOLwl5zy, zeQZkL1MKw]  [zeQZkL1MKw, gpMOLwl5zy]
453  [kRQabn8MKZ, pzeQZ6xQKw]  [pzeQZ6xQKw, kRQabn8MKZ]
598  [jYQJ8EW5GR, kRQabn8MKZ]  [kRQabn8MKZ, jYQJ8EW5GR]
642  [NWMWlBK5lz, a2lqR4JMr0]  [a2lqR4JMr0, NWMWlBK5lz]
701  [APk5LGOMOW, a2lqR4JMr0]  [a2lqR4JMr0, APk5LGOMOW]

In [26]:
def expand_gplus(df):
    expanded = df[["player_id", "data"]].explode("data")

    metrics = pd.json_normalize(expanded["data"].tolist())
    metrics["player_id"] = expanded["player_id"].values

    return metrics.set_index(
        ["player_id", "action_type"]
    ).sort_index()


single_metrics = expand_gplus(single)
multi_metrics = expand_gplus(multi)

differences = single_metrics.compare(multi_metrics)

print("Action records with differences:", len(differences))
display(differences.head(10))

Action records with differences: 0


player_id,action_type


In [ ]:
gk_gplus_2025 = asa.get_goalkeeper_goals_added(
    leagues="mls",
    season_name="2025",
    split_by_seasons=True
).to_pandas()

xg_2025 = asa.get_player_xgoals(
    leagues="mls",
    season_name="2025",
    split_by_seasons=True
).to_pandas()

xpass_2025 = asa.get_player_xpass(
    leagues="mls",
    season_name="2025",
    split_by_seasons=True
).to_pandas()

salaries_2025 = asa.get_player_salaries(
    leagues="mls",
    season_name="2025"
).to_pandas()

datasets = {
    "Goalkeeper g+": gk_gplus_2025,
    "xG": xg_2025,
    "xPass": xpass_2025,
    "Salaries": salaries_2025
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Unique players:", df["player_id"].nunique())
    print(
        "Unmatched IDs:",
        (~df["player_id"].isin(players["player_id"])).sum()
    )
    display(df.head(3))


Goalkeeper g+
Shape: (67, 5)
Unique players: 67
Unmatched IDs: 0


,minutes_played,player_id,team_id,season_name,data
0,2251,0Oq6woOxQ6,mKAqBBmqbg,2025,"[{'action_type': 'Claiming', 'goals_added_raw': 0.1842, 'goals_added_above_avg': 0.1941, 'count_actions': 31}, {'action_type': 'Fielding', 'goals_added_raw': -0.1873, 'goals_added_above_avg': -0.0653, 'count_actions': 383}, {'action_type': 'Handling', 'goals_added_raw': -0.3916, 'goals_added_above_avg': -0.3724, 'count_actions': 62}, {'action_type': 'Passing', 'goals_added_raw': 1.2441, 'goals_added_above_avg': -0.1556, 'count_actions': 690}, {'action_type': 'Shotstopping', 'goals_added_raw': -2.4809, 'goals_added_above_avg': -1.8683, 'count_actions': 105}, {'action_type': 'Sweeping', 'goals_added_raw': -0.2486, 'goals_added_above_avg': -0.2239, 'count_actions': 44}]"
1,3552,0x5g2Z8ZM7,NPqxKXZ59d,2025,"[{'action_type': 'Claiming', 'goals_added_raw': 0.0745, 'goals_added_above_avg': 0.0901, 'count_actions': 30}, {'action_type': 'Fielding', 'goals_added_raw': -0.2304, 'goals_added_above_avg': -0.0379, 'count_actions': 713}, {'action_type': 'Handling', 'goals_added_raw': 0.1664, 'goals_added_above_avg': 0.1967, 'count_actions': 136}, {'action_type': 'Passing', 'goals_added_raw': 2.036, 'goals_added_above_avg': -0.1727, 'count_actions': 1126}, {'action_type': 'Shotstopping', 'goals_added_raw': 6.2603, 'goals_added_above_avg': 7.2269, 'count_actions': 184}, {'action_type': 'Sweeping', 'goals_added_raw': -0.7213, 'goals_added_above_avg': -0.6823, 'count_actions': 42}]"
2,1039,0x5g2mYrM7,mKAqBBmqbg,2025,"[{'action_type': 'Claiming', 'goals_added_raw': 0.0691, 'goals_added_above_avg': 0.0737, 'count_actions': 30}, {'action_type': 'Fielding', 'goals_added_raw': -0.0581, 'goals_added_above_avg': -0.0018, 'count_actions': 182}, {'action_type': 'Handling', 'goals_added_raw': 0.5419, 'goals_added_above_avg': 0.5507, 'count_actions': 48}, {'action_type': 'Passing', 'goals_added_raw': 0.7607, 'goals_added_above_avg': 0.1146, 'count_actions': 375}, {'action_type': 'Shotstopping', 'goals_added_raw': 6.0108, 'goals_added_above_avg': 6.2935, 'count_actions': 61}, {'action_type': 'Sweeping', 'goals_added_raw': 0.2861, 'goals_added_above_avg': 0.2976, 'count_actions': 26}]"



xG
Shape: (854, 19)
Unique players: 854
Unmatched IDs: 0


,minutes_played,general_position,goals,xplace,goals_minus_xgoals,points_added,primary_assists,primary_assists_minus_xassists,xgoals,shots,season_name,shots_on_target,xassists,xgoals_plus_xassists,goals_plus_primary_assists,player_id,team_id,key_passes,xpoints_added
0,1634,FB,3,0.3516,1.8082,1.5481,2,0.1131,1.1918,12,2025,5,1.8869,3.0787,5,0Oq624oPq6,jYQJ19EqGR,8,0.6115
1,861,W,0,-1.1973,-1.951,0,0,-0.1121,1.951,13,2025,3,0.1121,2.0631,0,0Oq630dXQ6,pzeQZ6xQKw,3,1.7386
2,840,CM,0,-0.2725,-0.4212,0,2,0.9097,0.4212,12,2025,1,1.0903,1.5115,2,0Oq632k7Q6,jYQJ19EqGR,16,0.2295



xPass
Shape: (854, 14)
Unique players: 854
Unmatched IDs: 0


,minutes_played,general_position,avg_distance_yds,player_id,team_id,pass_completion_percentage,passes_completed_over_expected_p100,avg_vertical_distance_yds,share_team_touches,passes_completed_over_expected,count_games,season_name,xpass_completion_percentage,attempted_passes
0,1634,FB,16.5458,0Oq624oPq6,jYQJ19EqGR,0.8235,1.3532,1.5205,0.0655,10.7305,23,2025,0.8099,793
1,861,W,17.453,0Oq630dXQ6,pzeQZ6xQKw,0.7055,-5.6177,-1.0188,0.0269,-9.1569,18,2025,0.7617,163
2,840,CM,16.8972,0Oq632k7Q6,jYQJ19EqGR,0.8563,2.8513,1.7192,0.0614,18.8473,18,2025,0.8278,661



Salaries
Shape: (1848, 7)
Unique players: 997
Unmatched IDs: 0


,base_salary,player_id,team_id,guaranteed_compensation,season_name,mlspa_release,position
0,80622,0Oq624oPq6,jYQJ19EqGR,80622,2025,2025-05-23,D
1,80622,0Oq624oPq6,jYQJ19EqGR,80622,2025,2025-10-01,D
2,2100000,0Oq630dXQ6,pzeQZ6xQKw,2100000,2025,2025-05-23,F


CPU times: user 77.2 ms, sys: 25.6 ms, total: 103 ms
Wall time: 87.5 ms


In [32]:
gk_gplus_multi = asa.get_goalkeeper_goals_added(
    leagues="mls",
    season_name=["2024", "2025"],
    split_by_seasons=True
).to_pandas()

xg_multi = asa.get_player_xgoals(
    leagues="mls",
    season_name=["2024", "2025"],
    split_by_seasons=True
).to_pandas()

xpass_multi = asa.get_player_xpass(
    leagues="mls",
    season_name=["2024", "2025"],
    split_by_seasons=True
).to_pandas()

salaries_multi = asa.get_player_salaries(
    leagues="mls",
    season_name=["2024", "2025"]
).to_pandas()

multi_datasets = {
    "Outfield g+": gplus_multi,
    "Goalkeeper g+": gk_gplus_multi,
    "xG": xg_multi,
    "xPass": xpass_multi,
    "Salaries": salaries_multi
}

summary = []

for name, df in multi_datasets.items():
    summary.append({
        "dataset": name,
        "records": len(df),
        "2024_records": (df["season_name"].astype(str) == "2024").sum(),
        "2025_records": (df["season_name"].astype(str) == "2025").sum(),
        "duplicate_player_seasons": df.duplicated(
            ["player_id", "season_name"]
        ).sum(),
        "unmatched_ids": (
            ~df["player_id"].isin(players["player_id"])
        ).sum()
    })

display(pd.DataFrame(summary))

,dataset,records,2024_records,2025_records,duplicate_player_seasons,unmatched_ids
0,Outfield g+,1523,736,787,0,0
1,Goalkeeper g+,133,66,67,0,0
2,xG,1655,801,854,0,0
3,xPass,1655,801,854,0,0
4,Salaries,3634,1786,1848,1668,0


In [ ]:
gplus_2016 = asa.get_player_goals_added(
    leagues="mls",
    season_name="2016",
    split_by_seasons=True
).to_pandas()

gk_gplus_2016 = asa.get_goalkeeper_goals_added(
    leagues="mls",
    season_name="2016",
    split_by_seasons=True
).to_pandas()

xg_2016 = asa.get_player_xgoals(
    leagues="mls",
    season_name="2016",
    split_by_seasons=True
).to_pandas()

xpass_2016 = asa.get_player_xpass(
    leagues="mls",
    season_name="2016",
    split_by_seasons=True
).to_pandas()

salaries_2017 = asa.get_player_salaries(
    leagues="mls",
    season_name="2017"
).to_pandas()

historical_datasets = {
    "Outfield g+": gplus_2016,
    "Goalkeeper g+": gk_gplus_2016,
    "xG": xg_2016,
    "xPass": xpass_2016,
    "Salaries": salaries_2017
}

summary = []

for name, df in historical_datasets.items():
    summary.append({
        "dataset": name,
        "records": len(df),
        "columns": len(df.columns),
        "seasons": df["season_name"].unique().tolist(),
        "unique_players": df["player_id"].nunique(),
        "duplicate_player_seasons": df.duplicated(
            ["player_id", "season_name"]
        ).sum(),
        "unmatched_ids": (
            ~df["player_id"].isin(players["player_id"])
        ).sum(),
        "same_columns_as_2025": (
            set(df.columns) == set(multi_datasets[name].columns)
        )
    })

display(pd.DataFrame(summary))

,dataset,records,columns,seasons,unique_players,duplicate_player_seasons,unmatched_ids,same_columns_as_2025
0,Outfield g+,467,6,[2016],467,0,0,True
1,Goalkeeper g+,43,5,[2016],43,0,0,True
2,xG,510,19,[2016],510,0,0,True
3,xPass,510,14,[2016],510,0,0,True
4,Salaries,1268,7,[2017],676,592,0,True


CPU times: user 126 ms, sys: 19.3 ms, total: 145 ms
Wall time: 7.05 s


In [34]:
from pathlib import Path

test_dir = Path("../data/api_test")
test_dir.mkdir(parents=True, exist_ok=True)

test_path = test_dir / "gplus_2016.json"

gplus_2016.to_json(
    test_path,
    orient="records",
    indent=2,
    double_precision=15
)

gplus_reloaded = pd.read_json(
    test_path,
    orient="records",
    dtype=False
)

print("Original shape:", gplus_2016.shape)
print("Reloaded shape:", gplus_reloaded.shape)
print("File size (KB):", round(test_path.stat().st_size / 1024, 2))

display(gplus_reloaded.head(3))

Original shape: (467, 6)
Reloaded shape: (467, 6)
File size (KB): 496.63


,minutes_played,general_position,player_id,team_id,season_name,data
0,2463,CB,0Oq60APM6D,Vj58weDM8n,2016,"[{'action_type': 'Dribbling', 'goals_added_raw': -0.054, 'goals_added_above_avg': -0.1584, 'count_actions': 247}, {'action_type': 'Fouling', 'goals_added_raw': -0.35250000000000004, 'goals_added_above_avg': -0.2023, 'count_actions': 48}, {'action_type': 'Interrupting', 'goals_added_raw': 1.8319, 'goals_added_above_avg': 0.12250000000000001, 'count_actions': 361}, {'action_type': 'Passing', 'goals_added_raw': 1.0844, 'goals_added_above_avg': 0.021400000000000002, 'count_actions': 1194}, {'action_type': 'Receiving', 'goals_added_raw': 0.0213, 'goals_added_above_avg': -0.7809, 'count_actions': 819}, {'action_type': 'Shooting', 'goals_added_raw': 0.0, 'goals_added_above_avg': -0.3804, 'count_actions': 0}]"
1,1721,DM,0Oq60VgM6D,kaDQ0wRqEv,2016,"[{'action_type': 'Dribbling', 'goals_added_raw': -0.2948, 'goals_added_above_avg': -0.49510000000000004, 'count_actions': 173}, {'action_type': 'Fouling', 'goals_added_raw': -0.0859, 'goals_added_above_avg': -0.0634, 'count_actions': 50}, {'action_type': 'Interrupting', 'goals_added_raw': 1.071, 'goals_added_above_avg': -0.1226, 'count_actions': 198}, {'action_type': 'Passing', 'goals_added_raw': 0.7475, 'goals_added_above_avg': 0.123, 'count_actions': 769}, {'action_type': 'Receiving', 'goals_added_raw': 0.6863, 'goals_added_above_avg': -0.0613, 'count_actions': 582}, {'action_type': 'Shooting', 'goals_added_raw': 0.2326, 'goals_added_above_avg': -0.2058, 'count_actions': 11}]"
2,1756,CB,0Oq67EgQ6D,Z2vQ1xlqrA,2016,"[{'action_type': 'Dribbling', 'goals_added_raw': 0.125, 'goals_added_above_avg': 0.0829, 'count_actions': 292}, {'action_type': 'Fouling', 'goals_added_raw': -0.006200000000000001, 'goals_added_above_avg': 0.1635, 'count_actions': 22}, {'action_type': 'Interrupting', 'goals_added_raw': 0.912, 'goals_added_above_avg': -0.45, 'count_actions': 247}, {'action_type': 'Passing', 'goals_added_raw': 2.1075, 'goals_added_above_avg': 1.1424, 'count_actions': 1134}, {'action_type': 'Receiving', 'goals_added_raw': 0.528, 'goals_added_above_avg': 0.137, 'count_actions': 796}, {'action_type': 'Shooting', 'goals_added_raw': 0.1038, 'goals_added_above_avg': -0.09280000000000001, 'count_actions': 3}]"


In [35]:
import json

original_records = json.loads(
    gplus_2016.to_json(orient="records", double_precision=15)
)

with test_path.open("r", encoding="utf-8") as file:
    stored_records = json.load(file)

reloaded_records = json.loads(
    gplus_reloaded.to_json(orient="records", double_precision=15)
)

print("Storage integrity:", original_records == stored_records)
print("Round-trip integrity:", original_records == reloaded_records)

Storage integrity: True
Round-trip integrity: True


In [36]:
display(pd.DataFrame({
    "original": gplus_2016.dtypes,
    "reloaded": gplus_reloaded.dtypes
}))

for column in ["team_id", "data"]:
    print(f"\n{column}")
    print("Original:")
    print(gplus_2016[column].apply(type).value_counts())
    print("Reloaded:")
    print(gplus_reloaded[column].apply(type).value_counts())

,original,reloaded
minutes_played,int64,int64
general_position,str,str
player_id,str,str
team_id,object,object
season_name,str,str
data,object,object



team_id
Original:
team_id
<class 'str'>     453
<class 'list'>     14
Name: count, dtype: int64
Reloaded:
team_id
<class 'str'>     453
<class 'list'>     14
Name: count, dtype: int64

data
Original:
data
<class 'numpy.ndarray'>    467
Name: count, dtype: int64
Reloaded:
data
<class 'list'>    467
Name: count, dtype: int64


## Findings and acquisition decision

### API exploration

Using `itscalledsoccer==2.4.1`, we established that the ASA API:

- Supports league-wide and multi-season retrieval across all required datasets.
- Provides consistent player identifiers across the tested responses.
- Returns historical performance and salary data with consistent column structures in the tested seasons.
- Exposes additional information and greater numerical precision than observed in the interactive g+ table.

The API is a viable alternative to the interactive tables. However, technical viability does not necessarily justify replacing the existing datasets.

### Acquisition trade-offs

| Interactive tables                                | API                                                     |
| ------------------------------------------------- | ------------------------------------------------------- |
| Already acquired and organized for analysis.      | Requires additional acquisition and preparation.        |
| Published metrics are formatted and ready to use. | Provides more granular metrics and numerical precision. |
| Requires player identity resolution.              | Provides native player identifiers.                     |
| Suitable for a bounded retrospective study.       | Better suited to automated and recurring acquisition.   |

The primary advantage of the API for this study is referential integrity. However, a previous exploration identified only 29 ambiguous matches among 6,847 MLS-matched player-seasons. Identity resolution therefore appears manageable, although unique matches still require validation.

Replacing the interactive tables would introduce additional acquisition, storage and transformation work without an established analytical benefit.

### Decision: hybrid acquisition

Use the existing interactive tables as the primary source of performance and salary data, and the API to retrieve the player directory, resolve ambiguous identities and obtain additional information when necessary.

This approach minimizes unnecessary data engineering while retaining access to the API's advantages. ASA player IDs will serve as the common player identifier in the processed dataset.

### Limitations and reconsideration criteria

Dataset-wide numerical equivalence between the two sources has not been established. Identity matching, salary eligibility and historical player characteristics still require validation.

The acquisition decision may be reconsidered if identity resolution proves unreliable, the published metrics lack sufficient precision, additional variables become necessary, or the project requires recurring data updates.

A subsequent exploration of salaries, temporal coverage and player retention will inform these requirements.